# Agents

In [101]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [102]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [103]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
	index=index,
	llm_client=openai_client,
	instructions=instructions,
)

In [104]:
search_tool = {
	"type": "function",
	"function": {
		"name": "search",
		"description": "Search the FAQ database for entries matching the given query.",
		"parameters": {
			"type": "object",
			"properties": {
				"query": {
					"type": "string",
					"description": "Search query text to look up in the course FAQ."
				}
			},
			"required": ["query"],
			"additionalProperties": False
		}
	}
}

In [105]:
def search(query):
	boost_dict = {"question": 3.0, "section": 0.5}
	filter_dict = {"course": "llm-zoomcamp"}

	return index.search(
		query,
		num_results=5,
		boost_dict=boost_dict,
		filter_dict=filter_dict
	)

In [106]:
messages = [
  {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

In [107]:
response = openai_client.chat.completions.create(
	model="gpt-4o-mini",
	messages=messages,
	user="llm-zoomcamp",
	stream=False,
	tools=[search_tool]
)

In [108]:
import json

if response.choices[0].finish_reason == "tool_calls":
	message = response.choices[0].message
	function_call = response.choices[0].message.tool_calls[0].function
  
	if function_call.name == "search":
		# retrieve tool call params and call search function
		args = json.loads(function_call.arguments)
		results = search(**args)
		result_json = json.dumps(results, indent=2)
		
		# add the model response and tool call results to the message history
		messages.append(message)
		messages.append({
			"role": "tool",
			"tool_call_id": response.choices[0].message.tool_calls[0].id,
			"content": result_json
		})

		# send new prompt with updated message history
		response = openai_client.chat.completions.create(
			model="gpt-4o-mini",
			messages=messages,
			user="llm-zoomcamp",
			stream=False,
			tools=[search_tool]
		)

		print(response.choices[0].message.content)
	else:
		print("Unable to perform search")
else:
	print(response.choices[0].message.content)

Yes, you can still join the course! However, if you want to receive a certificate, make sure to submit your project while submissions are still being accepted. You can start learning and submitting homework right away, even without registering formally.


## Agentic loop

In [109]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [110]:
def make_call(call):
	if call.function.name == "search":
		args = json.loads(call.function.arguments)
		results = search(**args)
		result_json = json.dumps(results, indent=2)

		return {
			"role": "tool",
			"tool_call_id": call.id,
			"content": result_json
		}

In [ ]:
question = "I just discovered the course. Can I join it?"

In [ ]:
def agent_loop(instructions, question, model="gpt-4o-mini") -> str:
	messages = [
		{"role": "developer", "content": instructions},
		{"role": "user", "content": question},
	]

	last_answer = "Could not generate a response"

	while True:
		has_function_calls = False
		
		response = openai_client.chat.completions.create(
			model=model,
			messages=messages,
			user="llm-zoomcamp",
			stream=False,
			tools=[search_tool]
		)

		message = response.choices[0].message
		messages.append(message)
		
		for item in response.choices:
			if item.finish_reason == "tool_calls":
				tool_call = message.tool_calls[0]
				print("function_call:", tool_call.function.name, tool_call.function.arguments)
				call_output = make_call(tool_call)
				messages.append(call_output)
				has_function_calls = True
			elif item.finish_reason == "stop":
				print(item.message)
				last_answer = item.message.content
				print(item.message.content)
		
		if has_function_calls == False:
			break

	return last_answer

In [116]:
agent_loop(instructions, "How do I run Olama locally?")

function_call: search {"query":"run Olama locally"}
function_call: search {"query":"Olama installation setup"}


AttributeError: 'Choice' object has no attribute 'content'